# Accuracy and saturation from the shared run table

Loads `data/runs.parquet`, falling back to `data/runs.csv` if Parquet or its reader is unavailable. Run from the repository root or `acc_saturation/`; pass a path to `load_runs` to select a format explicitly.

Accuracy is the mean binary `successful` outcome per `agent_id` (full model/scaffold/parameter configuration). CORE-Bench v1.1 uses `main39`; Extended uses `ood19`. The `reliability` split is excluded.

Wilson and normal 95% intervals and the existing top-versus-fifth saturation formula (`alpha=0.5`) are retained. Saturation assumes equal sample sizes across configurations; ties are ordered by agent ID.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import math

pd.set_option("display.max_colwidth", None)

# Wilson CI
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denom
    return center - margin, center + margin


def load_runs(path=None):
    """Load the shared run table from either CSV or Parquet."""
    if path is None:
        root = next(
            (p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/runs.parquet").is_file() or (p / "data/runs.csv").is_file()),
            None,
        )
        if root is None:
            raise FileNotFoundError("Run from the repository or pass a path to runs.csv/runs.parquet.")
        path = root / "data/runs.parquet"
        if not path.is_file():
            path = path.with_suffix(".csv")
        if path.suffix == ".parquet":
            try:
                df = pd.read_parquet(path)
            except ImportError:
                path = path.with_suffix(".csv")
                df = pd.read_csv(path)
        else:
            df = pd.read_csv(path)
    else:
        path = Path(path)
        if path.suffix == ".parquet":
            df = pd.read_parquet(path)
        elif path.suffix == ".csv":
            df = pd.read_csv(path)
        else:
            raise ValueError("Expected a .csv or .parquet run table.")

    required = {"agent_id", "split", "successful"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Run table is missing columns: {sorted(missing)}")
    # Reject missing/invalid outcomes rather than silently counting them as failures.
    if df[list(required)].isna().any().any():
        raise ValueError("agent_id, split and successful must not contain missing values.")
    if not df["successful"].isin([True, False]).all():
        raise ValueError("successful must contain boolean or binary outcomes.")
    df["successful"] = df["successful"].astype(int)
    print(f"Loaded {len(df):,} runs from {path}")
    return df


# Compute accuracy for each full agent configuration within one benchmark split.
def compute_accuracy(df, split):
    subset = df.loc[df["split"] == split]
    if subset.empty:
        raise ValueError(f"No runs found for split {split!r}.")
    rows = []
    for agent, group in subset.groupby("agent_id"):
        n = len(group)
        k = group["successful"].sum()
        p = k / n
        lo, hi = wilson_ci(k, n)
        rows.append({"Agent": agent, "N": n, "Correct": int(k), "Accuracy": p, "CI_lower": lo, "CI_upper": hi})
    return pd.DataFrame(rows).sort_values("Agent").reset_index(drop=True)


def fmt_wilson(r):
    return f"{r.Accuracy:.1%}  [{r.CI_lower:.1%}, {r.CI_upper:.1%}]"


def fmt_normal(r):
    margin = 1.96 * np.sqrt(r.Accuracy * (1 - r.Accuracy) / r.N)
    return f"{r.Accuracy:.1%} ± {margin:.1%}"


def make_table(result, cols, fmt_fn):
    tbl = result.copy()
    tbl["Accuracy (95% CI)"] = result.apply(fmt_fn, axis=1)
    tbl = tbl[cols]
    tbl.index = range(1, len(tbl) + 1)
    return tbl

In [2]:
# Set an explicit path here to select CSV or Parquet instead of auto-detection.
runs = load_runs()
result = compute_accuracy(runs, "main39")
display(make_table(result, ["Agent", "Correct", "N", "Accuracy (95% CI)"], fmt_wilson))


Loaded 2,607 runs from /Users/nityanadgir/Downloads/corebench-analysis/data/runs.parquet


,Agent,Correct,N,Accuracy (95% CI)
1,corebench_hard_claude_code_agent_anthropic_claude_opus_4_5_max_thinking_tokens_10000,38,39,"97.4% [86.8%, 99.5%]"
2,corebench_hard_claude_code_agent_anthropic_claude_opus_4_6_baseline,38,39,"97.4% [86.8%, 99.5%]"
3,corebench_hard_codex_agent_gpt_5_1_reasoning_effort_medium,34,39,"87.2% [73.3%, 94.4%]"
4,corebench_hard_codex_agent_gpt_5_2_reasoning_effort_medium,37,39,"94.9% [83.1%, 98.6%]"
5,corebench_hard_codex_agent_gpt_5_3_codex_reasoning_effort_medium,38,39,"97.4% [86.8%, 99.5%]"
6,corebench_hard_codex_agent_gpt_5_4_reasoning_effort_high,38,39,"97.4% [86.8%, 99.5%]"
7,corebench_hard_codex_agent_gpt_5_4_reasoning_effort_low,36,39,"92.3% [79.7%, 97.3%]"
8,corebench_hard_codex_agent_gpt_5_4_reasoning_effort_medium,37,39,"94.9% [83.1%, 98.6%]"
9,corebench_hard_codex_agent_gpt_5_4_reasoning_effort_medium_max_threads_1,37,39,"94.9% [83.1%, 98.6%]"
10,corebench_hard_codex_agent_gpt_5_4_reasoning_effort_medium_max_threads_3,38,39,"97.4% [86.8%, 99.5%]"


In [3]:
display(make_table(result, ["Agent", "Accuracy (95% CI)"], fmt_normal))


,Agent,Accuracy (95% CI)
1,corebench_hard_claude_code_agent_anthropic_claude_opus_4_5_max_thinking_tokens_10000,97.4% ± 5.0%
2,corebench_hard_claude_code_agent_anthropic_claude_opus_4_6_baseline,97.4% ± 5.0%
3,corebench_hard_codex_agent_gpt_5_1_reasoning_effort_medium,87.2% ± 10.5%
4,corebench_hard_codex_agent_gpt_5_2_reasoning_effort_medium,94.9% ± 6.9%
5,corebench_hard_codex_agent_gpt_5_3_codex_reasoning_effort_medium,97.4% ± 5.0%
6,corebench_hard_codex_agent_gpt_5_4_reasoning_effort_high,97.4% ± 5.0%
7,corebench_hard_codex_agent_gpt_5_4_reasoning_effort_low,92.3% ± 8.4%
8,corebench_hard_codex_agent_gpt_5_4_reasoning_effort_medium,94.9% ± 6.9%
9,corebench_hard_codex_agent_gpt_5_4_reasoning_effort_medium_max_threads_1,94.9% ± 6.9%
10,corebench_hard_codex_agent_gpt_5_4_reasoning_effort_medium_max_threads_3,97.4% ± 5.0%


In [ ]:
# CORE-Bench Extended (out of distribution); reliability runs are excluded.
result_ood = compute_accuracy(runs, "ood19")
display(make_table(result_ood, ["Agent", "Correct", "N", "Accuracy (95% CI)"], fmt_wilson))


In [ ]:
# Calculate saturation index
def se_difference(acc_top, acc_k, n, alpha):
    se = np.sqrt(((acc_top*(1-acc_top))/(n**alpha)) + ((acc_k*(1-acc_k))/(n**alpha)))
    if se == 0:
        return 1
    else:
        return se

# Calculate statistical similarity
def statistical_similarity(acc_top, acc_k, n, alpha):
    delta = acc_top - acc_k
    difference = 1.96 * se_difference(acc_top, acc_k, n, alpha)
    return delta, difference

# Calculate score compression
def score_compression(acc_top, acc_k, n, alpha):
    return (acc_top - acc_k) / se_difference(acc_top, acc_k, n, alpha)

# Calculate saturation index
def saturation_index(acc_top, acc_k, n, alpha):
    return math.exp(-1 * (score_compression(acc_top, acc_k, n, alpha)))


In [ ]:
def saturation_summary(result, label, alpha=0.5):
    if len(result) < 5:
        print(f"{label}: saturation requires at least five agent configurations.")
        return
    if result["N"].nunique() != 1:
        raise ValueError("The saturation calculation requires equal run counts per agent.")
    ranked = result.sort_values(["Accuracy", "Agent"], ascending=[False, True]).reset_index(drop=True)
    top = ranked.iloc[0]
    fifth = ranked.iloc[4]
    n = top["N"]

    si = saturation_index(top["Accuracy"], fifth["Accuracy"], n, alpha)
    delta, margin = statistical_similarity(top["Accuracy"], fifth["Accuracy"], n, alpha)

    print(f"=== {label} ===")
    print(f"Top agent Accuracy:     {top['Accuracy']:.2%}")
    print(f"Fifth agent Accuracy:   {fifth['Accuracy']:.2%}")
    print(f"Saturation index:          {si:.4f}")
    print(f"Delta (top - fifth):       {delta:.4f}")
    print(f"1.96 * SE (similarity CI): {margin:.4f}")
    print(f"Statistically similar:     {delta <= margin}")
    print()

saturation_summary(result, "CORE-Bench v1.1 Saturation")
saturation_summary(result_ood, "CORE-Bench OOD Saturation")